# 02 - Data Preparation

## Objective
Prepare clean and structured datasets for further analysis in PostgreSQL and Power BI.

## Preparation Steps
1. Load raw datasets
2. Inspect data structure
3. Handle missing values and duplicates
4. Create analysis-ready datasets
5. Export processed datasets

In [28]:
import os
import pandas as pd
from pathlib import Path

# Check current working directory
print("Current directory:", os.getcwd())

# Check files and folders
print("\nCurrent folder contents:")
print(os.listdir())

Current directory: c:\Users\JEANNICE LAURENSIA S\OneDrive\Documents\Projects\Project 1\notebook

Current folder contents:
['01_data_exploration.ipynb', '02_data_preparation.ipynb', '03_product_analysis.sql']


## 1. Load Raw Datasets

Load the original Olist datasets into Pandas DataFrames.

The raw datasets will remain unchanged. We will create separate cleaned datasets during the preparation process.

In [29]:
# Define data directory
DATA_DIR = Path("../data")

# Load raw datasets
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [30]:
# Display dataset dimensions
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


## 2. Initial Data Quality Assessment

Assess the quality of the raw datasets before applying any transformations.

The assessment covers:
- Missing values
- Duplicate records
- Data types

In [31]:
# Check missing values in each dataset

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)


--- CUSTOMERS ---
No missing values.

--- ORDERS ---
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

--- ORDER_ITEMS ---
No missing values.

--- PAYMENTS ---
No missing values.

--- REVIEWS ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- PRODUCTS ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- SELLERS ---
No missing values.

--- GEOLOCATION ---
No missing values.

--- CATEGORY_TRANSLATION ---
No missing values.


In [32]:
# Check duplicated rows in each dataset

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count} duplicated rows")

customers: 0 duplicated rows
orders: 0 duplicated rows
order_items: 0 duplicated rows
payments: 0 duplicated rows
reviews: 0 duplicated rows
products: 0 duplicated rows
sellers: 0 duplicated rows
geolocation: 261831 duplicated rows
category_translation: 0 duplicated rows


In [33]:
# Display column data types

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(df.dtypes)


--- CUSTOMERS ---
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

--- ORDERS ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

--- ORDER_ITEMS ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

--- PAYMENTS ---
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

--- REVIEWS ---
review_id 

In [34]:
# Create a comprehensive data quality summary

quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "dataset": name,
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "missing_cells": df.isnull().sum().sum(),
        "columns_with_missing": (df.isnull().sum() > 0).sum(),
        "duplicate_rows": df.duplicated().sum()
    })

quality_summary = pd.DataFrame(quality_summary)

quality_summary

,dataset,total_rows,total_columns,missing_cells,columns_with_missing,duplicate_rows
0,customers,99441,5,0,0,0
1,orders,99441,8,4908,3,0
2,order_items,112650,7,0,0,0
3,payments,103886,5,0,0,0
4,reviews,99224,7,145903,2,0
5,products,32951,9,2448,8,0
6,sellers,3095,4,0,0,0
7,geolocation,1000163,5,0,0,261831
8,category_translation,71,2,0,0,0


In [35]:
# Detailed missing value analysis

for name, df in datasets.items():
    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if not missing.empty:
        missing_summary = pd.DataFrame({
            "missing_count": missing,
            "missing_percentage": (missing / len(df) * 100).round(2)
        })

        print(f"\n--- {name.upper()} ---")
        display(missing_summary)


--- ORDERS ---


,missing_count,missing_percentage
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98



--- REVIEWS ---


,missing_count,missing_percentage
review_comment_title,87656,88.34
review_comment_message,58247,58.70



--- PRODUCTS ---


,missing_count,missing_percentage
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [36]:
# Check order status distribution
print(orders["order_status"].value_counts(dropna=False))

# Compare missing timestamps by order status
timestamp_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for col in timestamp_columns:
    print(f"\n--- Missing {col} by order status ---")
    print(
        orders[orders[col].isna()]
        ["order_status"]
        .value_counts()
    )

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

--- Missing order_approved_at by order status ---
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

--- Missing order_delivered_carrier_date by order status ---
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

--- Missing order_delivered_customer_date by order status ---
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [37]:
# Inspect products with missing values
products_missing = products[products.isnull().any(axis=1)]

print("Rows with missing values:", len(products_missing))

display(products_missing.head(10))

Rows with missing values: 611


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [38]:
# Calculate unique geolocation records
geolocation_unique = geolocation.drop_duplicates()

print("Original rows:", len(geolocation))
print("Rows after removing exact duplicates:", len(geolocation_unique))
print("Remaining duplicates:", geolocation_unique.duplicated().sum())

Original rows: 1000163
Rows after removing exact duplicates: 738332
Remaining duplicates: 0


## 3. Data Cleaning and Transformation

Create separate cleaned copies of the raw datasets.

Cleaning decisions:
- Preserve all order statuses and missing timestamps.
- Convert order timestamp columns into datetime format.
- Replace missing product categories with "unknown".
- Preserve optional review comments and missing product attributes.
- Remove exact duplicate records from geolocation.

In [39]:
# Create independent copies of raw datasets

customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geolocation_clean = geolocation.copy()
category_translation_clean = category_translation.copy()

print("Clean dataset copies created successfully!")

Clean dataset copies created successfully!


In [40]:
# Identify timestamp columns
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert columns to datetime
for col in timestamp_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

# Check resulting data types
print(orders_clean[timestamp_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [41]:
# Replace missing product categories
products_clean["product_category_name"] = (
    products_clean["product_category_name"].fillna("unknown")
)

# Verify
print(
    "Remaining missing product categories:",
    products_clean["product_category_name"].isna().sum()
)

Remaining missing product categories: 0


In [42]:
# Remove exact duplicate records
geolocation_clean = geolocation_clean.drop_duplicates().copy()

# Verify
print("Original rows:", len(geolocation))
print("Clean rows:", len(geolocation_clean))
print("Remaining duplicates:", geolocation_clean.duplicated().sum())

Original rows: 1000163
Clean rows: 738332
Remaining duplicates: 0


In [43]:
# Compare raw and cleaned datasets

clean_datasets = {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "geolocation": geolocation_clean,
    "category_translation": category_translation_clean
}

for name, df in clean_datasets.items():
    print(
        f"{name}: {df.shape} | "
        f"duplicates: {df.duplicated().sum()}"
    )

customers: (99441, 5) | duplicates: 0
orders: (99441, 8) | duplicates: 0
order_items: (112650, 7) | duplicates: 0
payments: (103886, 5) | duplicates: 0
reviews: (99224, 7) | duplicates: 0
products: (32951, 9) | duplicates: 0
sellers: (3095, 4) | duplicates: 0
geolocation: (738332, 5) | duplicates: 0
category_translation: (71, 2) | duplicates: 0


## 4. Data Validation

Validate primary and foreign key relationships to ensure referential integrity across datasets.

In [44]:
# Define relationship checks

relationships = {
    "orders -> customers": {
        "left": orders_clean,
        "left_key": "customer_id",
        "right": customers_clean,
        "right_key": "customer_id"
    },
    "order_items -> orders": {
        "left": order_items_clean,
        "left_key": "order_id",
        "right": orders_clean,
        "right_key": "order_id"
    },
    "order_items -> products": {
        "left": order_items_clean,
        "left_key": "product_id",
        "right": products_clean,
        "right_key": "product_id"
    },
    "order_items -> sellers": {
        "left": order_items_clean,
        "left_key": "seller_id",
        "right": sellers_clean,
        "right_key": "seller_id"
    }
}

for name, relation in relationships.items():
    left = relation["left"]
    right = relation["right"]

    left_key = relation["left_key"]
    right_key = relation["right_key"]

    unmatched = ~left[left_key].isin(right[right_key])

    print(f"\n{name}")
    print("Unmatched records:", unmatched.sum())


orders -> customers
Unmatched records: 0

order_items -> orders
Unmatched records: 0

order_items -> products
Unmatched records: 0

order_items -> sellers
Unmatched records: 0


In [45]:
# Check primary key uniqueness

primary_keys = {
    "customers": (customers_clean, "customer_id"),
    "orders": (orders_clean, "order_id"),
    "products": (products_clean, "product_id"),
    "sellers": (sellers_clean, "seller_id")
}

for name, (df, key) in primary_keys.items():
    print(
        f"{name}:",
        "Unique =", df[key].is_unique,
        "| Missing =", df[key].isna().sum()
    )

customers: Unique = True | Missing = 0
orders: Unique = True | Missing = 0
products: Unique = True | Missing = 0
sellers: Unique = True | Missing = 0


In [46]:
# Create processed data directory

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Processed folder:", PROCESSED_DIR.resolve())

Processed folder: C:\Users\JEANNICE LAURENSIA S\OneDrive\Documents\Projects\Project 1\data\processed


In [47]:
# Export cleaned datasets to CSV

for name, df in clean_datasets.items():
    output_path = PROCESSED_DIR / f"{name}.csv"
    df.to_csv(output_path, index=False)

    print(f"Exported: {name}.csv | Rows: {len(df)}")

Exported: customers.csv | Rows: 99441
Exported: orders.csv | Rows: 99441
Exported: order_items.csv | Rows: 112650
Exported: payments.csv | Rows: 103886
Exported: reviews.csv | Rows: 99224
Exported: products.csv | Rows: 32951
Exported: sellers.csv | Rows: 3095
Exported: geolocation.csv | Rows: 738332
Exported: category_translation.csv | Rows: 71


In [48]:
# Check duplicate review IDs

print("Total rows:", len(reviews_clean))
print("Unique review IDs:", reviews_clean["review_id"].nunique())
print("Duplicate review IDs:", reviews_clean["review_id"].duplicated().sum())

Total rows: 99224
Unique review IDs: 98410
Duplicate review IDs: 814


In [49]:
# Display duplicate review records

duplicate_reviews = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].sort_values("review_id")

display(duplicate_reviews.head(10))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [50]:
# Check duplicate combinations of review_id and order_id

duplicate_pairs = reviews_clean.duplicated(
    subset=["review_id", "order_id"]
).sum()

print("Duplicate review-order pairs:", duplicate_pairs)

Duplicate review-order pairs: 0


In [51]:
# Check how many orders are associated with each review ID

review_order_counts = (
    reviews_clean.groupby("review_id")["order_id"]
    .nunique()
)

print("Review IDs linked to multiple orders:",
      (review_order_counts > 1).sum())

Review IDs linked to multiple orders: 789


In [52]:
# Convert product count/length columns to nullable integers

integer_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

for col in integer_columns:
    products_clean[col] = products_clean[col].astype("Int64")

# Verify data types
print(products_clean[integer_columns].dtypes)

product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
dtype: object


In [53]:
products_clean.to_csv(
    PROCESSED_DIR / "products.csv",
    index=False
)

print("products.csv exported successfully!")

products.csv exported successfully!


In [54]:
# Recalculate customer metrics using the same SQL logic

orders_check = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

customer_check = orders_check.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

print("Delivered orders:", orders_check["order_id"].nunique())
print("Customer IDs:", customer_check["customer_id"].nunique())
print("Unique customers:", customer_check["customer_unique_id"].nunique())
print("Missing customer IDs:", customer_check["customer_unique_id"].isna().sum())

Delivered orders: 96470
Customer IDs: 96470
Unique customers: 93350
Missing customer IDs: 0
